Calculate reaction time, sorting the correct trial and wrong trial.
only use for reaction time mode data.

In [1]:
# only use for reaction time (RT) analysis

# build_rt_tables.py
# ------------------------------------------------------------
# Build per-subject RT tables from single-trial gaze CSVs
# Modes:
#   1) Batch (default): scan all trials under GAZE_DIR
#   2) Single subject/file-key: --key <subject_id_or_filename_key>
#   3) Single trial file:      --gaze-file <path/to/trial_..._.csv>
#
# Outputs (per key):
#   A) subject_trial_rt/subject_<key>_rt.csv
#      columns: trial_index, GI_level, RT, cor_wro
#   B) subject_trial_rt_cor/subject_<key>_rt.csv
#      columns: trial_index, GI_level, RT  (cor==1 OR GI==0)
#   C) subject_trial_rt_wro/subject_<key>_rt.csv
#      columns: trial_index, GI_level, RT  (cor==0 OR GI==0)
#
# Robust to Jupyter/VSCode injected "-f/--f=<kernel.json>" arguments.
# ------------------------------------------------------------

from __future__ import annotations
import sys
import re
import argparse
from pathlib import Path
import pandas as pd

# ======== Default PATHS (edit if needed) ========
GAZE_DIR = r"Z:\BioMotionAnlyze\analyze\data\pymovement data\exp 202504\gaze_processed"
INFO_DIR = r"Z:\BioMotionAnlyze\analyze\data\meta data\exp 202504\trialInfo"

OUT_MERGED = r"Z:\BioMotionAnlyze\analyze\data\pymovement data\exp 202504\subject_trial_rt"
OUT_COR    = r"Z:\BioMotionAnlyze\analyze\data\pymovement data\exp 202504\subject_trial_rt_cor"
OUT_WRO    = r"Z:\BioMotionAnlyze\analyze\data\pymovement data\exp 202504\subject_trial_rt_wro"

# ------------------------------------------------------------
# 0) Strip Jupyter/VSCode kernel args so argparse never sees them
# ------------------------------------------------------------
def _strip_jupyter_args(argv=None):
    """Remove Jupyter/VSCode injected -f/--f arguments (may be '-f <file>' or '--f=<file>')."""
    if argv is None:
        argv = sys.argv
    out = [argv[0]]
    skip_next = False
    for a in argv[1:]:
        if skip_next:
            skip_next = False
            continue
        if a in ("-f", "--f"):
            skip_next = True
            continue
        if a.startswith("--f="):
            continue
        out.append(a)
    return out

sys.argv = _strip_jupyter_args()

# ======== Create output dirs ========
for p in (OUT_MERGED, OUT_COR, OUT_WRO):
    Path(p).mkdir(parents=True, exist_ok=True)

# ======== Filename pattern & helpers ========
TRIAL_RE = re.compile(r'^trial_(.+)_(\d+)\.csv$', re.IGNORECASE)

def parse_trial_filename(fname: str):
    """
    Accepts: trial_<key>_<trialid>.csv
      - <key> can be numeric subject id ('15') or long filename key ('2024-07-06(122)-r0003-eyeTrcAnlyz')
      - <trialid> is an integer and will be used as trial_index
    Returns (key:str, trial_id:int) or (None, None) if not matched.
    """
    m = TRIAL_RE.match(Path(fname).name)
    if not m:
        return None, None
    return m.group(1), int(m.group(2))

def safe_out_name(key: str) -> str:
    return f"subject_{key}_rt.csv".strip()

def find_info_file(key: str, info_dir: Path) -> Path | None:
    c = info_dir / f"subject_{key}_trialInfo.csv"
    return c if c.exists() else None

def extract_rt_and_gi(gaze_csv_path: Path):
    """
    RT  = last value in 'time' column.
    GI  = first non-null value in 'stimulus_index' (or fallback 'GI_level' if present).
    """
    df = pd.read_csv(gaze_csv_path)

    if "time" not in df.columns:
        raise ValueError(f"'time' column not found in {gaze_csv_path}")
    rt = float(df["time"].iloc[-1])

    if "stimulus_index" in df.columns:
        gi_series = df["stimulus_index"].dropna()
    elif "GI_level" in df.columns:
        gi_series = df["GI_level"].dropna()
    else:
        raise ValueError(f"Neither 'stimulus_index' nor 'GI_level' in {gaze_csv_path}")

    gi = float(gi_series.iloc[0]) if not gi_series.empty else float("nan")
    return rt, gi

def load_trialinfo(info_csv_path: Path) -> pd.DataFrame:
    """
    Load trialInfo and keep 'trial_index', 'cor_wro'.
    """
    ti = pd.read_csv(info_csv_path)
    for col in ("trial_index", "cor_wro"):
        if col not in ti.columns:
            raise ValueError(f"Column '{col}' missing in {info_csv_path}")
    ti = ti[["trial_index", "cor_wro"]].copy()
    ti["trial_index"] = ti["trial_index"].astype(int)
    ti["cor_wro"]     = ti["cor_wro"].astype(int)
    return ti

def write_outputs_for_key(key: str, rows: list[dict], append: bool):
    """
    Save:
      merged -> OUT_MERGED/subject_<key>_rt.csv
      correct -> OUT_COR/subject_<key>_rt.csv   (cor==1 OR GI==0)
      wrong   -> OUT_WRO/subject_<key>_rt.csv   (cor==0 OR GI==0)
    De-duplicate by trial_index if --append.
    """
    if not rows:
        print(f"[WARN] No rows to write for '{key}'.")
        return

    merged = pd.DataFrame(rows).sort_values("trial_index").reset_index(drop=True)

    merged_out = Path(OUT_MERGED) / safe_out_name(key)
    if append and merged_out.exists():
        old = pd.read_csv(merged_out)
        merged = pd.concat([old, merged], ignore_index=True)
        merged = merged.drop_duplicates(subset=["trial_index"], keep="last").sort_values("trial_index")

    # Write merged with cor_wro
    merged[["trial_index", "GI_level", "RT", "cor_wro"]].to_csv(merged_out, index=False, encoding="utf-8-sig")

    # Split views (GI==0 goes to BOTH correct and wrong)
    cor_mask = (merged["cor_wro"] == 1) | (merged["GI_level"] == 0)
    wro_mask = (merged["cor_wro"] == 0) | (merged["GI_level"] == 0)

    cor_df = merged.loc[cor_mask, ["trial_index", "GI_level", "RT"]].sort_values("trial_index")
    wro_df = merged.loc[wro_mask, ["trial_index", "GI_level", "RT"]].sort_values("trial_index")

    cor_out = Path(OUT_COR) / safe_out_name(key)
    wro_out = Path(OUT_WRO) / safe_out_name(key)

    if append:
        if cor_out.exists():
            old = pd.read_csv(cor_out)
            cor_df = (pd.concat([old, cor_df], ignore_index=True)
                        .drop_duplicates(subset=["trial_index"], keep="last")
                        .sort_values("trial_index"))
        if wro_out.exists():
            old = pd.read_csv(wro_out)
            wro_df = (pd.concat([old, wro_df], ignore_index=True)
                        .drop_duplicates(subset=["trial_index"], keep="last")
                        .sort_values("trial_index"))

    cor_df.to_csv(cor_out, index=False, encoding="utf-8-sig")
    wro_df.to_csv(wro_out, index=False, encoding="utf-8-sig")

    print(f"[OK] {key}: merged='{merged_out.name}', cor='{cor_out.name}', wro='{wro_out.name}'")

# ======== Processing modes ========
def process_key(key: str, gaze_dir: Path, info_dir: Path, append: bool):
    info_file = find_info_file(key, info_dir)
    if info_file is None:
        print(f"[WARN] trialInfo not found for key '{key}' in {info_dir}")
        return
    ti = load_trialinfo(info_file)

    rows = []
    for p in gaze_dir.glob(f"trial_{key}_*.csv"):
        _, trial_id = parse_trial_filename(p.name)
        if trial_id is None:
            continue
        try:
            rt, gi = extract_rt_and_gi(p)
        except Exception as e:
            print(f"[ERROR] {p}: {e}")
            continue
        hit = ti.loc[ti["trial_index"] == trial_id]
        cor = int(hit["cor_wro"].iloc[0]) if not hit.empty else pd.NA
        rows.append({"trial_index": trial_id, "GI_level": gi, "RT": rt, "cor_wro": cor})

    write_outputs_for_key(key, rows, append)

def process_single_file(gaze_file: Path, info_file: Path | None, info_dir: Path, append: bool):
    if not Path(gaze_file).exists():
        print(f"[WARN] Gaze file not found: {gaze_file}")
        return
    key, trial_id = parse_trial_filename(Path(gaze_file).name)
    if key is None:
        print(f"[WARN] File name not in pattern 'trial_<key>_<id>.csv': {gaze_file}")
        return
    if info_file is None:
        info_file = find_info_file(key, info_dir)
    if info_file is None or not Path(info_file).exists():
        print(f"[WARN] trialInfo not found for key '{key}'. You may pass --info-file explicitly.")
        return

    ti = load_trialinfo(info_file)
    try:
        rt, gi = extract_rt_and_gi(Path(gaze_file))
    except Exception as e:
        print(f"[ERROR] {gaze_file}: {e}")
        return

    hit = ti.loc[ti["trial_index"] == trial_id]
    cor = int(hit["cor_wro"].iloc[0]) if not hit.empty else pd.NA
    rows = [{"trial_index": trial_id, "GI_level": gi, "RT": rt, "cor_wro": cor}]
    write_outputs_for_key(key, rows, append)

def process_all(gaze_dir: Path, info_dir: Path, append: bool):
    keys = set()
    for p in Path(gaze_dir).glob("trial_*_*.csv"):
        k, _ = parse_trial_filename(p.name)
        if k is not None:
            keys.add(k)
    if not keys:
        print("[WARN] No trial files found.")
        return
    for k in sorted(keys):
        process_key(k, Path(gaze_dir), Path(info_dir), append)

# ======== CLI ========
def main():
    ap = argparse.ArgumentParser(description="Build RT tables (merged + COR/WRO split).")
    ap.add_argument("--gaze-dir", default=GAZE_DIR, help="Folder containing trial gaze CSVs.")
    ap.add_argument("--info-dir", default=INFO_DIR, help="Folder containing trialInfo CSVs.")
    ap.add_argument("--key", help="Process only this subject/file key (e.g., '15' or '2024-07-06(122)-r0003-eyeTrcAnlyz').")
    ap.add_argument("--gaze-file", help="Process a single gaze trial CSV file.")
    ap.add_argument("--info-file", help="Optional explicit trialInfo CSV for single-file mode.")
    ap.add_argument("--append", action="store_true", help="Append & dedupe by trial_index if outputs already exist.")

    # absorb ipykernel's '-f/--f' if it still sneaks in
    ap.add_argument("-f", "--f", help=argparse.SUPPRESS)

    # use parse_known_args to ignore any other unknown args safely
    args, _unknown = ap.parse_known_args()

    gaze_dir = Path(args.gaze_dir)
    info_dir = Path(args.info_dir)

    if args.gaze_file:
        process_single_file(Path(args.gaze_file), Path(args.info_file) if args.info_file else None, info_dir, args.append)
    elif args.key:
        process_key(args.key, gaze_dir, info_dir, args.append)
    else:
        process_all(gaze_dir, info_dir, args.append)

if __name__ == "__main__":
    main()


[OK] 11: merged='subject_11_rt.csv', cor='subject_11_rt.csv', wro='subject_11_rt.csv'
[OK] 111: merged='subject_111_rt.csv', cor='subject_111_rt.csv', wro='subject_111_rt.csv'
[OK] 112: merged='subject_112_rt.csv', cor='subject_112_rt.csv', wro='subject_112_rt.csv'
[OK] 113: merged='subject_113_rt.csv', cor='subject_113_rt.csv', wro='subject_113_rt.csv'
[OK] 114: merged='subject_114_rt.csv', cor='subject_114_rt.csv', wro='subject_114_rt.csv'
[OK] 115: merged='subject_115_rt.csv', cor='subject_115_rt.csv', wro='subject_115_rt.csv'
[OK] 116: merged='subject_116_rt.csv', cor='subject_116_rt.csv', wro='subject_116_rt.csv'
[OK] 117: merged='subject_117_rt.csv', cor='subject_117_rt.csv', wro='subject_117_rt.csv'
[OK] 118: merged='subject_118_rt.csv', cor='subject_118_rt.csv', wro='subject_118_rt.csv'
[OK] 119: merged='subject_119_rt.csv', cor='subject_119_rt.csv', wro='subject_119_rt.csv'
[OK] 12: merged='subject_12_rt.csv', cor='subject_12_rt.csv', wro='subject_12_rt.csv'
[OK] 120: merged='

event cor wro calculate

In [4]:
# build_event_tables_by_key.py
# ------------------------------------------------------------
# Read trialInfo files and per-trial event CSVs to build per-subject/per-key
# event tables, split by correctness. GI==0 is duplicated to BOTH sides.
# Supports:
#   - MODE="all"    : scan a folder of trialInfo files (subject_*_trialInfo.csv)
#   - MODE="single" : process ONE trialInfo file (SINGLE_TRIALINFO_FILE) or by SINGLE_KEY
# ------------------------------------------------------------

from __future__ import annotations
import os
from pathlib import Path
import re
import pandas as pd

# ======= PATHS =======
TRIALINFO_DIR = r"Z:\BioMotionAnlyze\analyze\data\meta data\202511_typeA\trialInfo\rt"
EVENTS_DIR    = r"Z:\BioMotionAnlyze\analyze\data\pymovement data\202511_typeA\rt\events_processed"

OUT_COR_DIR   = r"Z:\BioMotionAnlyze\analyze\data\pymovement data\202511_typeA\rt\subject_trial_event_cor"
OUT_WRO_DIR   = r"Z:\BioMotionAnlyze\analyze\data\pymovement data\202511_typeA\rt\subject_trial_event_wro"
# ======= MODE =======
MODE = "all"     # "all" or "single"
# -- single mode options (二选一，优先用文件路径) --
SINGLE_TRIALINFO_FILE = r""   # e.g. r"...\subject_2024-07-06(122)-r0003-eyeTrcAnlyz_trialInfo.csv"
SINGLE_KEY            = r""   # e.g. "2024-07-06(122)-r0003-eyeTrcAnlyz" or "15"

# ======= Helpers =======
TRIALINFO_GLOB = "subject_*_trialInfo.csv"  # 支持 subject_<数字>_ 或 subject_<filename>_

def key_from_trialinfo_filename(path: Path) -> str | None:
    """
    Extract <key> from 'subject_<key>_trialInfo.csv'.
    Works for numeric IDs and filename-like keys (with parentheses/dashes).
    """
    name = path.name
    m = re.match(r"^subject_(.+)_trialInfo\.csv$", name, flags=re.IGNORECASE)
    return m.group(1) if m else None

def load_event_stats(key: str, trial_id: int) -> dict:
    """
    Read events file: events_processed/trial_<key>_<trial_id>.csv
    Return counts & mean durations for fixation/saccade.
    Missing file -> zeros/NaN.
    """
    fpath = Path(EVENTS_DIR) / f"trial_{key}_{trial_id}.csv"
    if not fpath.exists():
        return {"fix_cnt": 0, "fix_mean": float("nan"), "sac_cnt": 0, "sac_mean": float("nan")}

    df = pd.read_csv(fpath)
    if df.empty:
        return {"fix_cnt": 0, "fix_mean": float("nan"), "sac_cnt": 0, "sac_mean": float("nan")}

    # normalize columns
    df.columns = [c.strip().lower() for c in df.columns]
    if "name" not in df.columns or "duration" not in df.columns:
        return {"fix_cnt": 0, "fix_mean": float("nan"), "sac_cnt": 0, "sac_mean": float("nan")}

    name = df["name"].astype(str).str.lower().str.strip()
    dur  = pd.to_numeric(df["duration"], errors="coerce")

    fix_mask = name.eq("fixation")
    sac_mask = name.eq("saccade")

    fix_cnt  = int(fix_mask.sum())
    sac_cnt  = int(sac_mask.sum())
    fix_mean = float(dur.where(fix_mask).mean()) if fix_cnt > 0 else float("nan")
    sac_mean = float(dur.where(sac_mask).mean()) if sac_cnt > 0 else float("nan")

    return {"fix_cnt": fix_cnt, "fix_mean": fix_mean, "sac_cnt": sac_cnt, "sac_mean": sac_mean}

def safe_float(x):
    try:
        return float(x)
    except Exception:
        return None

def process_one_trialinfo_file(ti_path: Path):
    key = key_from_trialinfo_filename(ti_path)
    if not key:
        print(f"[WARN] Skip (not a trialInfo file): {ti_path.name}")
        return

    df_info = pd.read_csv(ti_path)
    needed = ["trial_index", "GI_level", "cor_wro"]
    if any(col not in df_info.columns for col in needed):
        print(f"[WARN] {ti_path.name} missing columns {needed}; skipped.")
        return

    df_info = df_info.rename(columns={"trial_index": "trial_id"})
    df_info["trial_id"] = pd.to_numeric(df_info["trial_id"], errors="coerce").astype("Int64")
    df_info["GI_level"] = df_info["GI_level"].apply(safe_float)
    df_info["cor_wro"]  = pd.to_numeric(df_info["cor_wro"], errors="coerce").fillna(-1).astype(int)

    rows = []
    for _, r in df_info.iterrows():
        if pd.isna(r["trial_id"]):
            continue
        tid = int(r["trial_id"])
        gi  = r["GI_level"]
        cw  = int(r["cor_wro"])

        stats = load_event_stats(key, tid)
        rows.append({
            "trial_id": tid,
            "GI_level": gi,
            "fixation_count": stats["fix_cnt"],
            "fixation_mean_duration": stats["fix_mean"],
            "saccade_count": stats["sac_cnt"],
            "saccade_mean_duration": stats["sac_mean"],
            "cor_wro": cw
        })

    if not rows:
        print(f"[WARN] No rows for {ti_path.name}")
        return

    df_subj = pd.DataFrame(rows)

    # === split with GI==0 duplicated ===
    gi0 = df_subj["GI_level"] == 0
    to_cor = (df_subj["cor_wro"] == 1) | gi0
    to_wro = (df_subj["cor_wro"] == 0) | gi0

    keep_cols = [
        "trial_id", "GI_level",
        "fixation_count", "fixation_mean_duration",
        "saccade_count", "saccade_mean_duration"
    ]

    Path(OUT_COR_DIR).mkdir(parents=True, exist_ok=True)
    Path(OUT_WRO_DIR).mkdir(parents=True, exist_ok=True)

    out_name = f"subject_{key}_event.csv"

    df_subj.loc[to_cor, keep_cols].sort_values("trial_id").to_csv(
        Path(OUT_COR_DIR) / out_name, index=False, encoding="utf-8-sig"
    )
    df_subj.loc[to_wro, keep_cols].sort_values("trial_id").to_csv(
        Path(OUT_WRO_DIR) / out_name, index=False, encoding="utf-8-sig"
    )

    print(f"[OK] {key}: COR -> {out_name} | WRO -> {out_name}")

def main():
    if MODE.lower() == "single":
        if SINGLE_TRIALINFO_FILE:
            ti = Path(SINGLE_TRIALINFO_FILE)
        elif SINGLE_KEY:
            ti = Path(TRIALINFO_DIR) / f"subject_{SINGLE_KEY}_trialInfo.csv"
        else:
            print("[ERROR] MODE='single' but neither SINGLE_TRIALINFO_FILE nor SINGLE_KEY is set.")
            return
        if not ti.exists():
            print(f"[ERROR] trialInfo file not found: {ti}")
            return
        process_one_trialinfo_file(ti)

    elif MODE.lower() == "all":
        dirp = Path(TRIALINFO_DIR)
        if not dirp.is_dir():
            print(f"[ERROR] TRIALINFO_DIR not found: {dirp}")
            return
        files = sorted(dirp.glob(TRIALINFO_GLOB))
        if not files:
            print(f"[WARN] No files matched: {TRIALINFO_GLOB}")
            return
        for ti in files:
            process_one_trialinfo_file(ti)
    else:
        print(f"[ERROR] Unknown MODE: {MODE}. Use 'single' or 'all'.")

if __name__ == "__main__":
    main()


[OK] 2025-11-25(122)-r0025-eyeTrcAnlyz: COR -> subject_2025-11-25(122)-r0025-eyeTrcAnlyz_event.csv | WRO -> subject_2025-11-25(122)-r0025-eyeTrcAnlyz_event.csv
[OK] 2025-11-25(122)-r0029-eyeTrcAnlyz: COR -> subject_2025-11-25(122)-r0029-eyeTrcAnlyz_event.csv | WRO -> subject_2025-11-25(122)-r0029-eyeTrcAnlyz_event.csv
[OK] 2025-11-25(122)-r0032-eyeTrcAnlyz: COR -> subject_2025-11-25(122)-r0032-eyeTrcAnlyz_event.csv | WRO -> subject_2025-11-25(122)-r0032-eyeTrcAnlyz_event.csv
[OK] 2025-11-25(122)-r0037-eyeTrcAnlyz: COR -> subject_2025-11-25(122)-r0037-eyeTrcAnlyz_event.csv | WRO -> subject_2025-11-25(122)-r0037-eyeTrcAnlyz_event.csv
[OK] 2025-11-25(122)-r0040-eyeTrcAnlyz: COR -> subject_2025-11-25(122)-r0040-eyeTrcAnlyz_event.csv | WRO -> subject_2025-11-25(122)-r0040-eyeTrcAnlyz_event.csv
[OK] 2025-11-25(122)-r0043-eyeTrcAnlyz: COR -> subject_2025-11-25(122)-r0043-eyeTrcAnlyz_event.csv | WRO -> subject_2025-11-25(122)-r0043-eyeTrcAnlyz_event.csv
[OK] 2025-11-25(122)-r0048-eyeTrcAnlyz: 

Merging data by sbuject. Use for the subject finish many block in an experiment.

In [8]:
# merge csv files for A experiment

import os

# === Paths ===
# Folder containing the CSV files to merge
input_dir = r"Z:\BioMotionAnlyze\analyze\data\pymovement data\202511_typeA\rt\subject_trial_event_cor"
# Folder to save merged outputs
output_dir = r"Z:\BioMotionAnlyze\analyze\data\pymovement data\202511_typeA\rt\subject_single_trial_event_cor"

os.makedirs(output_dir, exist_ok=True)

def _open_with_fallback(path, mode):
    """
    Open a text file trying common encodings. 
    We prefer 'utf-8-sig' to preserve BOM if present; 
    fall back to 'utf-8' then 'gbk'. All with errors='ignore'.
    """
    encodings = ['utf-8-sig', 'utf-8', 'gbk']
    last_err = None
    for enc in encodings:
        try:
            return open(path, mode, encoding=enc, errors='ignore', newline='')
        except Exception as e:
            last_err = e
            continue
    # As a last resort, open in binary (rarely needed). Not used here because we need to skip first line easily.
    raise last_err

def list_csv_sorted():
    """Return a lexicographically sorted list of CSV filenames in input_dir."""
    return sorted([f for f in os.listdir(input_dir) if f.lower().endswith(".csv")])

def merge_file_group(file_list):
    """
    Merge one group of CSV files into a single CSV by raw line append:
      - write the entire first file (including its header)
      - for every subsequent file, skip its first line (header) then append the rest
    Output filename = <first_file_basename>_merged.csv
    """
    if not file_list:
        print("Empty file list; nothing to merge.")
        return None

    first_name = file_list[0]
    out_name = os.path.splitext(first_name)[0] + ".csv"
    out_path = os.path.join(output_dir, out_name)

    # Open output once in write mode (text)
    with _open_with_fallback(out_path, 'w') as fout:
        for idx, fname in enumerate(file_list):
            in_path = os.path.join(input_dir, fname)
            with _open_with_fallback(in_path, 'r') as fin:
                if idx == 0:
                    # Write the whole first file including header
                    for line in fin:
                        fout.write(line)
                else:
                    # Skip exactly one header line from subsequent files
                    _ = next(fin, None)  # discard the first line
                    for line in fin:
                        fout.write(line)

    print(f"Group merged -> {out_path}")
    return out_path

# === Mode 1: Group mode (every N consecutive files as one merged output) ===
def merge_in_groups(group_size):
    """
    Split all CSVs (sorted by name) into consecutive groups of 'group_size'
    and merge each group separately.
      e.g., 9 files with group_size=3 -> groups [0:3], [3:6], [6:9]
    """
    files = list_csv_sorted()
    if not files:
        print("No CSV files found in input directory.")
        return []

    outputs = []
    for i in range(0, len(files), group_size):
        group = files[i:i+group_size]
        out = merge_file_group(group)
        if out:
            outputs.append(out)
    return outputs

# === Mode 2: Specified files mode (merge only the files you list, in that order) ===
def merge_specified(file_list):
    """
    Merge only the specified filenames (must exist in input_dir), in the given order,
    keeping only the header of the first file.
    """
    if not file_list:
        print("Specified list is empty.")
        return None

    # Optional: verify existence and warn if any is missing
    missing = [f for f in file_list if not os.path.exists(os.path.join(input_dir, f))]
    if missing:
        print("Warning: some specified files were not found and will be skipped:")
        for m in missing:
            print(" -", m)
        file_list = [f for f in file_list if os.path.exists(os.path.join(input_dir, f))]

    if not file_list:
        print("No valid files to merge after filtering.")
        return None

    return merge_file_group(file_list)

# ===== How to run (choose ONE) =====
# 1) Group mode: every 3 files as a group (examples: 1–3, 4–6, 7–9)
merge_in_groups(3)

# 2) Specified mode: only merge the files you name (in order)
# merge_specified([
#     "file1.csv",
#     "file2.csv",
#     "file3.csv"
# ])

Group merged -> Z:\BioMotionAnlyze\analyze\data\pymovement data\202511_typeA\rt\subject_single_trial_event_cor\subject_2025-11-25(122)-r0025-eyeTrcAnlyz_event.csv
Group merged -> Z:\BioMotionAnlyze\analyze\data\pymovement data\202511_typeA\rt\subject_single_trial_event_cor\subject_2025-11-25(122)-r0037-eyeTrcAnlyz_event.csv
Group merged -> Z:\BioMotionAnlyze\analyze\data\pymovement data\202511_typeA\rt\subject_single_trial_event_cor\subject_2025-11-25(122)-r0048-eyeTrcAnlyz_event.csv
Group merged -> Z:\BioMotionAnlyze\analyze\data\pymovement data\202511_typeA\rt\subject_single_trial_event_cor\subject_2025-11-26(122)-r0004-eyeTrcAnlyz_event.csv
Group merged -> Z:\BioMotionAnlyze\analyze\data\pymovement data\202511_typeA\rt\subject_single_trial_event_cor\subject_2025-11-26(122)-r0024-eyeTrcAnlyz_event.csv
Group merged -> Z:\BioMotionAnlyze\analyze\data\pymovement data\202511_typeA\rt\subject_single_trial_event_cor\subject_2025-12-05(122)-r0007-eyeTrcAnlyz_event.csv
Group merged -> Z:\Bio

['Z:\\BioMotionAnlyze\\analyze\\data\\pymovement data\\202511_typeA\\rt\\subject_single_trial_event_cor\\subject_2025-11-25(122)-r0025-eyeTrcAnlyz_event.csv',
 'Z:\\BioMotionAnlyze\\analyze\\data\\pymovement data\\202511_typeA\\rt\\subject_single_trial_event_cor\\subject_2025-11-25(122)-r0037-eyeTrcAnlyz_event.csv',
 'Z:\\BioMotionAnlyze\\analyze\\data\\pymovement data\\202511_typeA\\rt\\subject_single_trial_event_cor\\subject_2025-11-25(122)-r0048-eyeTrcAnlyz_event.csv',
 'Z:\\BioMotionAnlyze\\analyze\\data\\pymovement data\\202511_typeA\\rt\\subject_single_trial_event_cor\\subject_2025-11-26(122)-r0004-eyeTrcAnlyz_event.csv',
 'Z:\\BioMotionAnlyze\\analyze\\data\\pymovement data\\202511_typeA\\rt\\subject_single_trial_event_cor\\subject_2025-11-26(122)-r0024-eyeTrcAnlyz_event.csv',
 'Z:\\BioMotionAnlyze\\analyze\\data\\pymovement data\\202511_typeA\\rt\\subject_single_trial_event_cor\\subject_2025-12-05(122)-r0007-eyeTrcAnlyz_event.csv',
 'Z:\\BioMotionAnlyze\\analyze\\data\\pymoveme

Meaning rt data by GI.

In [6]:
# rt_mean_by_gi_modes_outdir.py
# Compute mean RT grouped by GI_level.
# Modes:
#   - MODE = "single": process ONE file (SINGLE_FILE)
#   - MODE = "all"   : process ALL files in a folder (INPUT_DIR + GLOB_PATTERN)
#
# Output for each input is written to OUT_DIR as "<input_stem>_byGI.csv"
# Columns: GI_level, mean_RT (no rounding)

from pathlib import Path
import pandas as pd

# ======= CONFIG =======
MODE = "all"  # "single" or "all"

# For MODE="single"
SINGLE_FILE = r"Z:\BioMotionAnlyze\analyze\data\pymovement data\test\subject_trial_rt\subject_15_rt.csv"

# For MODE="all"
INPUT_DIR = r"Z:\BioMotionAnlyze\analyze\data\pymovement data\exp 202504\subject_trial_rt_wro"
GLOB_PATTERN = "subject_*_rt.csv"   # which files to process under INPUT_DIR

# ======= OUTPUT DIR (required) =======
OUT_DIR = r"Z:\BioMotionAnlyze\analyze\data\pymovement data\exp 202504\subject_rt by GI_wro"
# ======= CORE =======
def process_one_file(in_path: Path, out_dir: Path):
    """Compute mean RT per GI_level for a single CSV and save to OUT_DIR/<stem>_byGI.csv."""
    try:
        df = pd.read_csv(in_path)
    except Exception as e:
        print(f"[ERROR] Cannot read {in_path}: {e}")
        return

    needed = {"GI_level", "RT"}
    missing = needed - set(df.columns)
    if missing:
        print(f"[WARN] Missing columns {missing} in {in_path.name}. Skipped.")
        return

    # Ensure numeric & drop NaNs
    df["GI_level"] = pd.to_numeric(df["GI_level"], errors="coerce")
    df["RT"] = pd.to_numeric(df["RT"], errors="coerce")
    work = df.dropna(subset=["GI_level", "RT"])
    if work.empty:
        print(f"[WARN] No valid rows in {in_path.name}. Skipped.")
        return

    # Group by GI and compute mean RT (no rounding)
    out = (work.groupby("GI_level", as_index=False)["RT"]
                .mean()
                .rename(columns={"RT": "mean_RT"})
                .sort_values("GI_level"))

    out_dir_path = Path(out_dir)
    out_dir_path.mkdir(parents=True, exist_ok=True)
    out_path = out_dir_path / f"{in_path.stem}_byGI.csv"
    try:
        out.to_csv(out_path, index=False, encoding="utf-8-sig")
        print(f"[OK] {in_path.name} -> {out_path}")
    except Exception as e:
        print(f"[ERROR] Cannot write {out_path}: {e}")

def main():
    out_dir = Path(OUT_DIR)

    if MODE.lower() == "single":
        p = Path(SINGLE_FILE)
        if not p.is_file():
            print(f"[ERROR] SINGLE_FILE not found: {p}")
            return
        process_one_file(p, out_dir)

    elif MODE.lower() == "all":
        d = Path(INPUT_DIR)
        if not d.is_dir():
            print(f"[ERROR] INPUT_DIR not found: {d}")
            return
        files = sorted(d.glob(GLOB_PATTERN))
        if not files:
            print(f"[WARN] No files matching '{GLOB_PATTERN}' in {d}")
            return
        for f in files:
            process_one_file(f, out_dir)

    else:
        print(f"[ERROR] Unknown MODE: {MODE}. Use 'single' or 'all'.")

if __name__ == "__main__":
    main()


[OK] subject_111_rt.csv -> Z:\BioMotionAnlyze\analyze\data\pymovement data\exp 202504\subject_rt by GI_wro\subject_111_rt_byGI.csv
[OK] subject_112_rt.csv -> Z:\BioMotionAnlyze\analyze\data\pymovement data\exp 202504\subject_rt by GI_wro\subject_112_rt_byGI.csv
[OK] subject_113_rt.csv -> Z:\BioMotionAnlyze\analyze\data\pymovement data\exp 202504\subject_rt by GI_wro\subject_113_rt_byGI.csv
[OK] subject_114_rt.csv -> Z:\BioMotionAnlyze\analyze\data\pymovement data\exp 202504\subject_rt by GI_wro\subject_114_rt_byGI.csv
[OK] subject_115_rt.csv -> Z:\BioMotionAnlyze\analyze\data\pymovement data\exp 202504\subject_rt by GI_wro\subject_115_rt_byGI.csv
[OK] subject_116_rt.csv -> Z:\BioMotionAnlyze\analyze\data\pymovement data\exp 202504\subject_rt by GI_wro\subject_116_rt_byGI.csv
[OK] subject_117_rt.csv -> Z:\BioMotionAnlyze\analyze\data\pymovement data\exp 202504\subject_rt by GI_wro\subject_117_rt_byGI.csv
[OK] subject_118_rt.csv -> Z:\BioMotionAnlyze\analyze\data\pymovement data\exp 2025

Meaning event by GI.

In [15]:
# event_mean_by_gi_modes.py
# Compute GI-wise means for event metrics per file.
# Modes:
#   - MODE = "single": process ONE CSV (SINGLE_FILE)
#   - MODE = "all"   : process ALL CSVs under INPUT_DIR matching GLOB_PATTERN
#
# Input columns required: GI_level, fixation_count, fixation_mean_duration,
#                         saccade_count, saccade_mean_duration
# Output: OUT_DIR / "<input_stem>_byGI.csv"

from pathlib import Path
import pandas as pd

# ======= CONFIG =======
MODE = "all"  # "single" or "all"

# Single-file mode
SINGLE_FILE = r"Z:\BioMotionAnlyze\analyze\data\pymovement data\test\subject_trial_event_cor\subject_15_event.csv"

# Batch mode
INPUT_DIR    = r"Z:\BioMotionAnlyze\analyze\data\pymovement data\202511_typeA\ft\subject_trial_event_cor"
GLOB_PATTERN = "subject_*_event.csv"   # matches both subject_15_event.csv and subject_<filename>_event.csv

# Output folder (ALWAYS write results here)
OUT_DIR = r"Z:\BioMotionAnlyze\analyze\data\pymovement data\202511_typeA\ft\subject_event_byGI_cor"
# If you want to process WRO files, set:
# INPUT_DIR = r"...\subject_trial_event_wro"
# OUT_DIR   = r"...\subject_event_byGI_wro"

REQUIRED_COLS = [
    "GI_level",
    "fixation_count", "fixation_mean_duration",
    "saccade_count", "saccade_mean_duration",
]

def process_one_file(in_path: Path, out_dir: Path):
    """Compute GI-wise means for one CSV and save to OUT_DIR/<stem>_byGI.csv."""
    try:
        df = pd.read_csv(in_path)
    except Exception as e:
        print(f"[ERROR] Cannot read {in_path}: {e}")
        return

    # Check required columns
    missing = [c for c in REQUIRED_COLS if c not in df.columns]
    if missing:
        print(f"[WARN] {in_path.name} missing columns {missing}. Skipped.")
        return

    # Coerce to numeric (robust to strings), drop NaN rows on GI_level
    df["GI_level"] = pd.to_numeric(df["GI_level"], errors="coerce")
    for c in REQUIRED_COLS[1:]:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    work = df.dropna(subset=["GI_level"])

    if work.empty:
        print(f"[WARN] No valid GI rows in {in_path.name}. Skipped.")
        return

    # Group by GI_level and compute means (no rounding)
    grouped = (
        work.groupby("GI_level", as_index=False)[
            ["fixation_count", "fixation_mean_duration",
             "saccade_count", "saccade_mean_duration"]
        ].mean()
        .sort_values("GI_level")
    )

    out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / f"{in_path.stem}_byGI.csv"

    try:
        grouped.to_csv(out_path, index=False, encoding="utf-8-sig")
        print(f"[OK] {in_path.name} -> {out_path}")
    except Exception as e:
        print(f"[ERROR] Cannot write {out_path}: {e}")

def main():
    out_dir = Path(OUT_DIR)

    if MODE.lower() == "single":
        p = Path(SINGLE_FILE)
        if not p.is_file():
            print(f"[ERROR] SINGLE_FILE not found: {p}")
            return
        process_one_file(p, out_dir)

    elif MODE.lower() == "all":
        d = Path(INPUT_DIR)
        if not d.is_dir():
            print(f"[ERROR] INPUT_DIR not found: {d}")
            return
        files = sorted(d.glob(GLOB_PATTERN))  # catch both numeric-ID and filename keys
        if not files:
            print(f"[WARN] No files matching '{GLOB_PATTERN}' in {d}")
            return
        for f in files:
            process_one_file(f, out_dir)
    else:
        print(f"[ERROR] Unknown MODE: {MODE}. Use 'single' or 'all'.")

if __name__ == "__main__":
    main()


[OK] subject_2025-11-25(122)-r0024-eyeTrcAnlyz_event.csv -> Z:\BioMotionAnlyze\analyze\data\pymovement data\202511_typeA\ft\subject_event_byGI_cor\subject_2025-11-25(122)-r0024-eyeTrcAnlyz_event_byGI.csv
[OK] subject_2025-11-25(122)-r0028-eyeTrcAnlyz_event.csv -> Z:\BioMotionAnlyze\analyze\data\pymovement data\202511_typeA\ft\subject_event_byGI_cor\subject_2025-11-25(122)-r0028-eyeTrcAnlyz_event_byGI.csv
[OK] subject_2025-11-25(122)-r0031-eyeTrcAnlyz_event.csv -> Z:\BioMotionAnlyze\analyze\data\pymovement data\202511_typeA\ft\subject_event_byGI_cor\subject_2025-11-25(122)-r0031-eyeTrcAnlyz_event_byGI.csv
[OK] subject_2025-11-25(122)-r0035-eyeTrcAnlyz_event.csv -> Z:\BioMotionAnlyze\analyze\data\pymovement data\202511_typeA\ft\subject_event_byGI_cor\subject_2025-11-25(122)-r0035-eyeTrcAnlyz_event_byGI.csv
[OK] subject_2025-11-25(122)-r0039-eyeTrcAnlyz_event.csv -> Z:\BioMotionAnlyze\analyze\data\pymovement data\202511_typeA\ft\subject_event_byGI_cor\subject_2025-11-25(122)-r0039-eyeTrcA